# ACD-LR: Benchmarking Deep Learning for Anatomical Completeness Detection

In low-resource medical settings, incomplete or corrupted CT scans waste time and resources.
Rather than just classifying a scan as "clean" or "corrupted" (binary), we frame the problem as
**3D segmentation**: given a CT volume, predict a voxel-wise map showing *where* the corruption is.

We use the [LUNA16 dataset](https://zenodo.org/records/3723295) (~888 scans) and synthetically
introduce six corruption types that model real-world failure modes in resource-constrained clinics.
Three segmentation architectures are benchmarked end-to-end.

**Corruption types** (motivated by [Masoudi et al. 2021](https://doi.org/10.1016/j.compmedimag.2020.101846)
and [Gu et al. 2021](https://doi.org/10.1016/j.patcog.2021.108109)):

| Corruption | What it models | Map type |
|---|---|---|
| Anatomical cropping | Incomplete scan range (operator error) | Thresholded voxel diff |
| Motion blur | Patient movement during acquisition | Thresholded voxel diff |
| Shift / jump | Positioning error or table shift | Thresholded voxel diff |
| Anatomical slicing | Data loss during transfer / storage | Thresholded voxel diff |
| Gaussian noise | Aging / low-quality detector electronics | Thresholded voxel diff |
| Resolution downsampling | Low-cost scanner / lossy compression | Thresholded voxel diff |

**Models** (all from [MONAI](https://monai.io/)):
- 3D U-Net (Ronneberger et al. 2015, extended to 3D by Cicek et al. 2016)
- 3D Residual U-Net (deeper encoder with residual connections)
- 3D Attention U-Net (Oktay et al. 2018 - attention gates for better localization)

In [ ]:
!pip -q install torch torchvision torchaudio monai SimpleITK scikit-learn

In [ ]:
import time
NOTEBOOK_START = time.time()

import os, glob, random, shutil
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import SimpleITK as sitk
from scipy.ndimage import zoom, gaussian_filter

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import CosineAnnealingLR

import monai
from monai.networks.nets import UNet, AttentionUnet
from monai.losses import DiceCELoss

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}  |  PyTorch {torch.__version__}  |  MONAI {monai.__version__}")

## Configuration

All hyperparameters and paths in one place so nothing is buried in the code.

In [ ]:
# paths (Kaggle layout)
DATA_DIR = "/kaggle/input/datasets/fardinabdullaacanto"
MASK_DIR = "/kaggle/input/datasets/gprem09/seg-lungs-luna16/seg-lungs-LUNA16"
OUT_DIR  = Path("./dataset_3d_maps")

# preprocessing
VOL_SHAPE  = (64, 128, 128)   # D x H x W after resize
HU_MIN     = -1000            # lung window lower bound
HU_MAX     = 400              # lung window upper bound

# corruption parameters
SLICE_FRAC       = (0.15, 0.35)   # fraction of depth to remove
SHIFT_PX         = (20, 35)       # pixel shift magnitude
BLUR_KERNEL      = (3, 9)         # motion blur kernel (odd values)
NOISE_STD        = (30.0, 80.0)   # Gaussian noise std in HU
DOWNSAMPLE_RANGE = (0.3, 0.6)     # resolution reduction factor
COMBO_WEIGHTS    = {"single": 0.55, "double": 0.30, "triple": 0.15}

ALL_CORRUPTIONS = [
    "crop", "blur", "shift", "slicing", "noise", "downsample"
]

# data split
TRAIN_FRAC = 0.70
VAL_FRAC   = 0.15

# training
BATCH    = 4
EPOCHS   = 50
LR       = 1e-3
WD       = 1e-5
MODELS   = ["unet", "resunet", "attn_unet"]

## Data Discovery

LUNA16 ships as 10 subsets of `.mhd`/`.raw` pairs. We glob through a few common
Kaggle directory layouts to find them all.

In [ ]:
scan_paths = []

for i in range(10):
    for pattern in [
        os.path.join(DATA_DIR, f"subset{i}", f"subset{i}"),
        os.path.join(DATA_DIR, f"subset{i}", f"subset{i}", f"subset{i}"),
        os.path.join(DATA_DIR, f"subset{i}"),
    ]:
        hits = sorted(glob.glob(os.path.join(pattern, "*.mhd")))
        if hits:
            scan_paths.extend(hits)
            print(f"  subset{i}: {len(hits):3d} scans")
            break
    else:
        hits = sorted(glob.glob(os.path.join(DATA_DIR, f"subset{i}", "**", "*.mhd"), recursive=True))
        scan_paths.extend(hits)
        print(f"  subset{i}: {len(hits):3d} scans (recursive)")

print(f"\nTotal scans found: {len(scan_paths)}")

## Preprocessing

Standard CT preprocessing pipeline following the approach in
[Hofmanninger et al. 2020](https://doi.org/10.1007/s10278-020-00341-1):
1. Optional lung-mask application (zeros out non-lung tissue)
2. HU clipping to the lung window [-1000, 400]
3. Min-max normalization to [0, 1]
4. Trilinear resize to a fixed 64 x 128 x 128 grid

In [ ]:
def load_scan(path):
    return sitk.GetArrayFromImage(sitk.ReadImage(path)).astype(np.float32)

def find_lung_mask(scan_path, mask_dir):
    """Try to find a matching mask file by filename."""
    fname = os.path.basename(scan_path)
    hits = glob.glob(os.path.join(mask_dir, "**", fname), recursive=True)
    if hits:
        return sitk.GetArrayFromImage(sitk.ReadImage(hits[0])).astype(np.float32)
    return None

def resize_vol(vol, target=VOL_SHAPE, order=1):
    return zoom(vol, [t/s for t, s in zip(target, vol.shape)], order=order).astype(np.float32)

def preprocess(vol_hu, mask=None):
    """
    Returns (processed_volume, resized_mask).
    If no mask available we treat every voxel as anatomy.
    """
    if mask is not None:
        vol_hu = np.where(mask > 0, vol_hu, HU_MIN)  # zero out non-lung
        msk = resize_vol(mask, VOL_SHAPE, order=0)
    else:
        msk = np.ones(VOL_SHAPE, dtype=np.float32)

    vol = np.clip(vol_hu, HU_MIN, HU_MAX)
    vol = ((vol - HU_MIN) / (HU_MAX - HU_MIN)).astype(np.float32)
    vol = resize_vol(vol, VOL_SHAPE)
    return vol, msk

In [ ]:
# quick sanity check
if scan_paths:
    _v = load_scan(scan_paths[0])
    _m = find_lung_mask(scan_paths[0], MASK_DIR)
    _vp, _mp = preprocess(_v, _m)
    print(f"Raw shape: {_v.shape}  ->  Processed: {_vp.shape}  range [{_vp.min():.2f}, {_vp.max():.2f}]")
    if _m is not None:
        print("Lung masks available - will isolate lung region before processing")
    else:
        print("No lung masks found - using full volume (this is fine, masks are optional)")

## Corruption Functions

Each function takes a preprocessed volume + its resized lung mask and returns
`(corrupted_volume, ground_truth_3d_map)`. The ground-truth map is derived
from the **per-voxel absolute difference** between the original and corrupted
volumes, thresholded at mean + 1 std. This gives anatomically meaningful maps
that highlight exactly where the corruption changed the scan, regardless of
whether lung masks are available.

- **Crop**: zeroed slices create large differences where anatomy existed.
- **Slicing**: removed-then-resized slices show interpolation artifacts.
- **Blur / shift / noise / downsampling**: difference highlights regions
  where the corruption had the strongest effect.

In [ ]:
def corrupt_crop(vol, mask, mode=None, rng=None):
    """Zero out a contiguous block of slices (simulates incomplete scan range)."""
    rng = rng or np.random.default_rng()
    d, h, w = vol.shape
    out = vol.copy()
    n = max(1, int(d * rng.uniform(*SLICE_FRAC)))
    if mode is None:
        mode = rng.choice(["top", "bottom", "both", "center", "scattered"])

    if mode == "top":
        out[:n] = 0
    elif mode == "bottom":
        out[-n:] = 0
    elif mode == "both":
        t, b = n // 2, n - n // 2
        out[:t] = 0; out[-b:] = 0
    elif mode == "center":
        lo = int(d * 0.25)
        hi = max(lo, int(d * 0.75) - n)
        s = rng.integers(lo, hi + 1)
        out[s:s+n] = 0
    else:  # scattered
        idx = rng.choice(d, size=n, replace=False)
        out[idx] = 0

    cmap = (np.abs(vol - out) > 1e-6).astype(np.float32)
    return out, cmap

In [ ]:
def corrupt_blur(vol, mask, rng=None):
    """Directional Gaussian blur (patient movement during acquisition)."""
    rng = rng or np.random.default_rng()
    ks = rng.choice(range(BLUR_KERNEL[0], BLUR_KERNEL[1] + 1, 2))
    axis = rng.choice(["axial", "coronal", "sagittal"])
    sigma = {
        "axial":    (ks/6, 0.5, 0.5),
        "coronal":  (0.5, ks/6, 0.5),
        "sagittal": (0.5, 0.5, ks/6),
    }[axis]

    blurred = gaussian_filter(vol, sigma=sigma, mode="nearest").astype(np.float32)
    diff = np.abs(vol - blurred)
    cmap = (diff > diff.mean() + diff.std()).astype(np.float32)
    return blurred, cmap

In [ ]:
def corrupt_shift(vol, mask, rng=None):
    """Rigid pixel shift across all slices (table/positioning error)."""
    rng = rng or np.random.default_rng()
    dx = int(rng.integers(*SHIFT_PX)) * rng.choice([-1, 1])
    dy = int(rng.integers(*SHIFT_PX)) * rng.choice([-1, 1])

    shifted = np.stack([np.roll(vol[z], (dy, dx), axis=(0,1)) for z in range(vol.shape[0])])
    diff = np.abs(vol - shifted)
    cmap = (diff > diff.mean() + diff.std()).astype(np.float32)
    return shifted, cmap

In [ ]:
def corrupt_slicing(vol, mask, rng=None):
    """Remove slices then resize back (data loss during transfer/archival)."""
    rng = rng or np.random.default_rng()
    d, h, w = vol.shape
    n = max(1, min(int(d * rng.uniform(*SLICE_FRAC)), d - 1))
    mode = rng.choice(["top", "bottom", "both", "middle", "scattered"])

    keep = np.ones(d, dtype=bool)
    if mode == "top":       keep[:n] = False
    elif mode == "bottom":  keep[-n:] = False
    elif mode == "both":
        t = n // 2; keep[:t] = False; keep[-(n-t):] = False
    elif mode == "middle":
        lo = int(d * 0.25)
        hi = max(lo, int(d * 0.75) - n)
        s = rng.integers(lo, hi + 1)
        keep[s:s+n] = False
    else:
        keep[rng.choice(d, size=n, replace=False)] = False

    restored = resize_vol(vol[keep], vol.shape, order=1)
    diff = np.abs(vol - restored)
    cmap = (diff > diff.mean() + diff.std()).astype(np.float32)
    return restored, cmap

In [ ]:
def corrupt_noise(vol, mask, rng=None):
    """Additive Gaussian noise (aging detector electronics)."""
    rng = rng or np.random.default_rng()
    std = rng.uniform(*NOISE_STD) / (HU_MAX - HU_MIN)
    noisy = np.clip(vol + rng.normal(0, std, vol.shape).astype(np.float32), 0, 1)
    diff = np.abs(vol - noisy)
    cmap = (diff > diff.mean() + diff.std()).astype(np.float32)
    return noisy, cmap

In [ ]:
def corrupt_downsample(vol, mask, rng=None):
    """Down-then-up resampling (low-cost scanner / lossy compression)."""
    rng = rng or np.random.default_rng()
    f = rng.uniform(*DOWNSAMPLE_RANGE)
    small_shape = tuple(max(1, int(s * f)) for s in vol.shape)
    small  = zoom(vol, [s/o for s, o in zip(small_shape, vol.shape)], order=1)
    back   = zoom(small, [o/s for o, s in zip(vol.shape, small.shape)], order=1)
    if back.shape != vol.shape:
        back = back[tuple(slice(0, s) for s in vol.shape)]
    back = back.astype(np.float32)
    diff = np.abs(vol - back)
    cmap = (diff > diff.mean() + diff.std()).astype(np.float32)
    return back, cmap

In [ ]:
DISPATCH = {
    "crop":       corrupt_crop,
    "blur":       corrupt_blur,
    "shift":      corrupt_shift,
    "slicing":    corrupt_slicing,
    "noise":      corrupt_noise,
    "downsample": corrupt_downsample,
}

def random_corruption(vol, mask, rng=None):
    """
    Randomly pick single / double / triple combo based on COMBO_WEIGHTS,
    apply sequentially, union the maps.
    """
    rng = rng or np.random.default_rng()

    keys = list(COMBO_WEIGHTS.keys())
    probs = np.array([COMBO_WEIGHTS[k] for k in keys])
    probs /= probs.sum()
    level = rng.choice(keys, p=probs)
    n = {"single": 1, "double": 2, "triple": 3}[level]

    picked = list(rng.choice(ALL_CORRUPTIONS, size=n, replace=False))
    # structural corruptions first (they change which slices exist)
    picked.sort(key=lambda x: 0 if x in ("crop", "slicing") else 1)

    combined = np.zeros_like(vol, dtype=np.float32)
    for name in picked:
        vol, cmap = DISPATCH[name](vol, mask, rng=rng)
        combined = np.maximum(combined, cmap)

    tag = picked[0] if n == 1 else f"combo_{n}"
    return vol, combined, tag

## Corruption Visualization

Side-by-side comparison of each corruption type on a single scan. The third and fourth columns
show the absolute voxel difference between the clean and corrupted volume (axial and coronal views),
following the visualization approach in [Masoudi et al. 2021](https://doi.org/10.1016/j.compmedimag.2020.101846).

In [ ]:
if scan_paths:
    _vol = load_scan(scan_paths[0])
    _msk = find_lung_mask(scan_paths[0], MASK_DIR)
    _pp, _mr = preprocess(_vol, _msk)

    fig, axes = plt.subplots(len(DISPATCH), 4, figsize=(16, 3.5 * len(DISPATCH)))
    fig.suptitle("Corruption Types and Their Effects", fontsize=15, y=1.01)

    for row, (name, fn) in enumerate(DISPATCH.items()):
        c_vol, c_map = fn(_pp.copy(), _mr.copy(), rng=np.random.default_rng(SEED))
        diff_vol = np.abs(_pp.astype(np.float64) - c_vol.astype(np.float64))

        # pick a slice where the corruption is actually visible
        per_slice_diff = diff_vol.mean(axis=(1, 2))
        if per_slice_diff.max() > 1e-4:
            z = int(np.argmax(per_slice_diff))
        else:
            z = _pp.shape[0] // 2

        axes[row, 0].imshow(_pp[z], cmap="gray", vmin=0, vmax=1)
        axes[row, 0].set_ylabel(name, fontsize=11, rotation=0, labelpad=55)
        if row == 0: axes[row, 0].set_title("Original")
        axes[row, 0].axis("off")

        axes[row, 1].imshow(c_vol[z], cmap="gray", vmin=0, vmax=1)
        if row == 0: axes[row, 1].set_title("Corrupted")
        axes[row, 1].axis("off")

        axes[row, 2].imshow(diff_vol[z], cmap="hot")
        if row == 0: axes[row, 2].set_title("Abs. Difference")
        axes[row, 2].axis("off")

        mid_c = diff_vol.shape[1] // 2
        axes[row, 3].imshow(diff_vol[:, mid_c, :], cmap="hot",
                            origin="lower", aspect="auto")
        if row == 0: axes[row, 3].set_title("Abs. Diff (coronal)")
        axes[row, 3].axis("off")

    plt.tight_layout()
    plt.show()

## Dataset Construction

For each scan we randomly decide clean (50 %) vs corrupted (50 %).
Corrupted samples get a random single/double/triple combo.
Everything is saved as compressed `.npz` with keys `vol`, `cmap`, `ctype`.

In [ ]:
def build_split(paths, split, seed=0):
    rng = np.random.default_rng(seed)
    out = OUT_DIR / split
    out.mkdir(parents=True, exist_ok=True)
    stats = {}

    for i, p in enumerate(paths):
        try:
            vol_hu = load_scan(p)
            mask   = find_lung_mask(p, MASK_DIR)
            vol, msk = preprocess(vol_hu, mask)

            if rng.random() < 0.5:
                vol, cmap, tag = random_corruption(vol, msk, rng)
            else:
                cmap = np.zeros_like(vol, dtype=np.float32)
                tag  = "clean"

            stats[tag] = stats.get(tag, 0) + 1
            np.savez_compressed(str(out / f"{split}_{i:04d}.npz"),
                                vol=vol, cmap=cmap.astype(np.float32), ctype=tag)

            if (i + 1) % 50 == 0 or i + 1 == len(paths):
                print(f"  [{split}] {i+1}/{len(paths)}")
        except Exception as e:
            print(f"  [{split}] ERROR {p}: {e}")

    print(f"  [{split}] done  ->  {stats}")
    return stats

In [ ]:
if OUT_DIR.exists():
    shutil.rmtree(OUT_DIR)

random.seed(SEED)
all_paths = scan_paths.copy()
random.shuffle(all_paths)

n = len(all_paths)
n_tr = int(n * TRAIN_FRAC)
n_va = int(n * VAL_FRAC)

tr_paths = all_paths[:n_tr]
va_paths = all_paths[n_tr:n_tr + n_va]
te_paths = all_paths[n_tr + n_va:]
print(f"Split: {len(tr_paths)} train / {len(va_paths)} val / {len(te_paths)} test\n")

build_split(tr_paths, "train", seed=SEED)
build_split(va_paths, "val",   seed=SEED + 1)
build_split(te_paths, "test",  seed=SEED + 2)

## DataLoaders

In [ ]:
class MapDataset(Dataset):
    def __init__(self, folder):
        self.files = sorted(Path(folder).glob("*.npz"))
        assert self.files, f"No .npz in {folder}"

    def __len__(self):  return len(self.files)

    def __getitem__(self, i):
        d = np.load(self.files[i])
        x = d["vol"].astype(np.float32)[None]   # (1,D,H,W)
        y = d["cmap"].astype(np.float32)[None]
        return torch.from_numpy(x), torch.from_numpy(y)

train_ds = MapDataset(OUT_DIR / "train")
val_ds   = MapDataset(OUT_DIR / "val")
test_ds  = MapDataset(OUT_DIR / "test")

train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_ds,  batch_size=BATCH, shuffle=False, num_workers=0)

xb, yb = next(iter(train_loader))
print(f"Sizes - train {len(train_ds)}  val {len(val_ds)}  test {len(test_ds)}")
print(f"Batch shapes:  x {tuple(xb.shape)}   y {tuple(yb.shape)}")

In [ ]:
fig, axes = plt.subplots(min(4, BATCH), 4, figsize=(14, 14))
fig.suptitle("Sample Training Pairs", fontsize=14)

for i in range(min(4, xb.shape[0])):
    v, m = xb[i,0].numpy(), yb[i,0].numpy()
    z = v.shape[0] // 2
    c = v.shape[1] // 2

    axes[i,0].imshow(v[z], cmap="gray"); axes[i,0].axis("off")
    axes[i,1].imshow(m[z], cmap="Reds", vmin=0, vmax=1); axes[i,1].axis("off")
    axes[i,2].imshow(v[:,c,:], cmap="gray", origin="lower", aspect="auto"); axes[i,2].axis("off")
    axes[i,3].imshow(m[:,c,:], cmap="Reds", vmin=0, vmax=1, origin="lower", aspect="auto"); axes[i,3].axis("off")

    if i == 0:
        for j, t in enumerate(["Volume (axial)", "Map (axial)", "Volume (coronal)", "Map (coronal)"]):
            axes[0,j].set_title(t)

plt.tight_layout()
plt.show()

## Model Definitions

All models output `(B, 1, D, H, W)` raw logits.  We use MONAI implementations
which handle 3D convolutions, batch-norm, and skip connections.

- **U-Net**: Cicek et al., "3D U-Net: Learning Dense Volumetric Segmentation
  from Sparse Annotation", MICCAI 2016.
- **Res-UNet**: same topology but with residual units for better gradient flow
  (He et al. 2016).
- **Attention U-Net**: Oktay et al., "Attention U-Net: Learning Where to Look
  for the Pancreas", MIDL 2018 - attention gates weigh skip connections.

In [ ]:
def make_model(name):
    if name == "unet":
        return UNet(spatial_dims=3, in_channels=1, out_channels=1,
                    channels=(32, 64, 128, 256), strides=(2, 2, 2), num_res_units=2)
    elif name == "resunet":
        return UNet(spatial_dims=3, in_channels=1, out_channels=1,
                    channels=(32, 64, 128, 256, 512), strides=(2, 2, 2, 2), num_res_units=3)
    elif name == "attn_unet":
        return AttentionUnet(spatial_dims=3, in_channels=1, out_channels=1,
                             channels=(32, 64, 128, 256), strides=(2, 2, 2))
    raise ValueError(name)

nparams = lambda m: sum(p.numel() for p in m.parameters())

# verify forward pass for each architecture
dummy = torch.randn(2, 1, *VOL_SHAPE, device=device)
for name in MODELS:
    m = make_model(name).to(device)
    out = m(dummy)
    print(f"  {name:12s}  params {nparams(m):>10,}   output {tuple(out.shape)}")
    del m
torch.cuda.empty_cache() if torch.cuda.is_available() else None

## Training

We use **Dice + Cross-Entropy loss** (DiceCELoss from MONAI) which balances
region overlap and per-voxel accuracy - standard for medical-image segmentation
([Isensee et al. 2021, nnU-Net](https://doi.org/10.1038/s41592-020-01008-z)).

Optimizer: AdamW with cosine-annealing LR schedule.

In [ ]:
def dice_score(pred, target, eps=1e-5):
    p = pred.reshape(-1)
    t = target.reshape(-1)
    inter = (p * t).sum()
    return float((2 * inter + eps) / (p.sum() + t.sum() + eps))

def iou_score(pred, target, eps=1e-5):
    p = pred.reshape(-1)
    t = target.reshape(-1)
    inter = (p * t).sum()
    return float((inter + eps) / (p.sum() + t.sum() - inter + eps))


def train_epoch(model, loader, opt, loss_fn):
    model.train()
    tot_loss, tot_dice, cnt = 0., 0., 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        opt.zero_grad()
        logits = model(x)
        loss = loss_fn(logits, y)
        loss.backward()
        opt.step()

        pred = (torch.sigmoid(logits) >= 0.5).float()
        tot_loss += loss.item() * x.size(0)
        tot_dice += dice_score(pred, y) * x.size(0)
        cnt += x.size(0)
    return tot_loss / cnt, tot_dice / cnt


@torch.no_grad()
def eval_epoch(model, loader, loss_fn):
    model.eval()
    tot_loss, tot_dice, tot_iou, cnt = 0., 0., 0., 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        loss = loss_fn(logits, y)

        pred = (torch.sigmoid(logits) >= 0.5).float()
        tot_loss += loss.item() * x.size(0)
        tot_dice += dice_score(pred, y) * x.size(0)
        tot_iou  += iou_score(pred, y) * x.size(0)
        cnt += x.size(0)
    return tot_loss / cnt, tot_dice / cnt, tot_iou / cnt

In [ ]:
def run_training(name):
    print(f"\n{'='*60}\n  {name}\n{'='*60}")
    model = make_model(name).to(device)
    loss_fn   = DiceCELoss(sigmoid=True, lambda_dice=1.0, lambda_ce=1.0)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
    sched     = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

    hist = {k: [] for k in ["tl", "td", "vl", "vd", "vi"]}
    best_dice, best_ep = 0., 0

    for ep in range(1, EPOCHS + 1):
        tl, td = train_epoch(model, train_loader, optimizer, loss_fn)
        vl, vd, vi = eval_epoch(model, val_loader, loss_fn)
        sched.step()

        hist["tl"].append(tl); hist["td"].append(td)
        hist["vl"].append(vl); hist["vd"].append(vd); hist["vi"].append(vi)

        if vd > best_dice:
            best_dice, best_ep = vd, ep
            torch.save(model.state_dict(), f"best_{name}.pth")

        if ep % 10 == 0 or ep == 1:
            print(f"  ep {ep:3d}  loss {tl:.4f}  dice {td:.3f}  |  val {vl:.4f}  dice {vd:.3f}  iou {vi:.3f}")

    print(f"  -> best val dice {best_dice:.4f}  @ epoch {best_ep}")
    del model
    torch.cuda.empty_cache() if torch.cuda.is_available() else None
    return {"hist": hist, "best_dice": best_dice, "best_ep": best_ep}

In [ ]:
results = {}
for name in MODELS:
    results[name] = run_training(name)

print("\n" + "="*60 + "\n  Training complete\n" + "="*60)
for n in MODELS:
    print(f"  {n:14s}  best dice {results[n]['best_dice']:.4f}  (ep {results[n]['best_ep']})")

## Training Curves

In [ ]:
clrs = {"unet": "#1f77b4", "resunet": "#ff7f0e", "attn_unet": "#2ca02c"}
fig, ax = plt.subplots(1, 3, figsize=(17, 4.5))

for n in MODELS:
    h = results[n]["hist"]
    ep = range(1, len(h["tl"]) + 1)
    c = clrs[n]
    ax[0].plot(ep, h["tl"], "--", color=c, alpha=.4, label=f"{n} tr")
    ax[0].plot(ep, h["vl"], color=c, label=f"{n} val")
    ax[1].plot(ep, h["td"], "--", color=c, alpha=.4, label=f"{n} tr")
    ax[1].plot(ep, h["vd"], color=c, label=f"{n} val")
    ax[2].plot(ep, h["vi"], color=c, label=n)

for a in ax:
    a.legend(fontsize=7); a.set_xlabel("Epoch"); a.grid(alpha=.3)
ax[0].set_title("DiceCE Loss"); ax[1].set_title("Dice Score"); ax[2].set_title("Val IoU")
plt.tight_layout(); plt.show()

## Test-Set Evaluation

In [ ]:
test_res = {}
loss_fn = DiceCELoss(sigmoid=True, lambda_dice=1.0, lambda_ce=1.0)

for n in MODELS:
    m = make_model(n).to(device)
    m.load_state_dict(torch.load(f"best_{n}.pth", map_location=device))
    tl, td, ti = eval_epoch(m, test_loader, loss_fn)
    test_res[n] = {"dice": td, "iou": ti, "loss": tl, "params": nparams(m)}
    print(f"  {n:14s}  dice {td:.4f}  iou {ti:.4f}  loss {tl:.4f}")
    del m

torch.cuda.empty_cache() if torch.cuda.is_available() else None

print("\n" + "="*65)
print(f"  {'Model':<16} {'Dice':>7} {'IoU':>7} {'Loss':>9} {'Params':>11}")
print("-"*60)
for n in MODELS:
    r = test_res[n]
    print(f"  {n:<16} {r['dice']:>7.4f} {r['iou']:>7.4f} {r['loss']:>9.4f} {r['params']:>11,}")
print("="*65)

## Baseline Comparison

To verify the segmentation models are learning something meaningful we compare against
a **trivial baseline**: threshold the per-voxel intensity. Since cropped regions are
set to zero and noise/blur change the intensity distribution, a simple "flag any voxel
below threshold *t*" already catches some corruption. If the learned models can't beat
this, the whole approach is questionable.

In [ ]:
# Baseline 1: flag voxels below an intensity threshold
# (catches cropped/zeroed slices but not noise or blur)

best_thr, best_baseline_dice = 0, 0
for thr in np.arange(0.01, 0.20, 0.01):
    ds_sum, cnt = 0., 0
    for xb, yb in test_loader:
        pred_b = (xb < thr).float()   # anything dark = "corrupted"
        for j in range(xb.size(0)):
            ds_sum += dice_score(pred_b[j], yb[j])
            cnt += 1
    avg = ds_sum / cnt
    if avg > best_baseline_dice:
        best_baseline_dice, best_thr = avg, thr

# Baseline 2: predict all-ones (upper bound for global corruptions,
# but terrible for clean samples - gives a sense of class imbalance)
ones_dice, cnt = 0., 0
for xb, yb in test_loader:
    for j in range(xb.size(0)):
        ones_dice += dice_score(torch.ones_like(yb[j]), yb[j])
        cnt += 1
all_ones_dice = ones_dice / cnt

# Baseline 3: predict all-zeros (lower bound)
zeros_dice, cnt = 0., 0
for xb, yb in test_loader:
    for j in range(xb.size(0)):
        zeros_dice += dice_score(torch.zeros_like(yb[j]), yb[j])
        cnt += 1
all_zeros_dice = zeros_dice / cnt

print(f"{'Baseline':<28} {'Dice':>8}")
print("-" * 38)
print(f"{'All-zeros (predict clean)':<28} {all_zeros_dice:>8.4f}")
print(f"{'All-ones (predict corrupt)':<28} {all_ones_dice:>8.4f}")
print(f"{'Intensity threshold (t={best_thr:.2f})':<28} {best_baseline_dice:>8.4f}")
print()
for n in MODELS:
    print(f"{n:<28} {test_res[n]['dice']:>8.4f}")
print()
gain = test_res[max(test_res, key=lambda k: test_res[k]['dice'])]['dice'] - best_baseline_dice
print(f"Best model beats best baseline by {gain:+.4f} Dice")

## Per-Corruption-Type Breakdown

How well does each model localize each specific corruption?

In [ ]:
te_npz = sorted((OUT_DIR / "test").glob("*.npz"))
te_types = [str(np.load(p, allow_pickle=True)["ctype"]) for p in te_npz]
uniq = sorted(set(te_types))

header = f"{'corruption':<22}"
for n in MODELS: header += f"  {n:>12}"
header += f"  {'count':>6}"
print(header + "\n" + "-" * len(header))

for ct in uniq:
    idxs = [i for i, t in enumerate(te_types) if t == ct]
    row = f"{ct:<22}"
    for mn in MODELS:
        model = make_model(mn).to(device)
        model.load_state_dict(torch.load(f"best_{mn}.pth", map_location=device))
        model.eval()
        ds = 0.
        for idx in idxs:
            d = np.load(te_npz[idx])
            xv = torch.from_numpy(d["vol"][None, None].astype(np.float32)).to(device)
            yv = torch.from_numpy(d["cmap"][None, None].astype(np.float32)).to(device)
            with torch.no_grad():
                p = (torch.sigmoid(model(xv)) >= 0.5).float()
            ds += dice_score(p, yv)
        row += f"  {ds/len(idxs):>12.4f}"
        del model
    row += f"  {len(idxs):>6}"
    print(row)

torch.cuda.empty_cache() if torch.cuda.is_available() else None

## Per-Sample Dice Distributions

Averages can hide a lot. The box plots below show per-sample Dice spread
for each model, broken down by corruption type. A tight distribution
means consistent performance; long tails mean some samples are getting
mislocalized badly.

In [ ]:
# collect per-sample dice for every (model, corruption_type) pair
per_sample = {n: {} for n in MODELS}

for mn in MODELS:
    model = make_model(mn).to(device)
    model.load_state_dict(torch.load(f"best_{mn}.pth", map_location=device))
    model.eval()

    for idx, npz_path in enumerate(te_npz):
        d = np.load(npz_path)
        ct = str(d["ctype"])
        xv = torch.from_numpy(d["vol"][None, None].astype(np.float32)).to(device)
        yv = torch.from_numpy(d["cmap"][None, None].astype(np.float32)).to(device)
        with torch.no_grad():
            p = (torch.sigmoid(model(xv)) >= 0.5).float()
        ds = dice_score(p, yv)
        per_sample[mn].setdefault(ct, []).append(ds)
    del model

torch.cuda.empty_cache() if torch.cuda.is_available() else None

# box plot: rows = corruption types, columns = models
fig, axes = plt.subplots(1, len(MODELS), figsize=(6 * len(MODELS), 5), sharey=True)
if len(MODELS) == 1: axes = [axes]

for i, mn in enumerate(MODELS):
    labels = sorted(per_sample[mn].keys())
    data   = [per_sample[mn][l] for l in labels]
    bp = axes[i].boxplot(data, labels=labels, patch_artist=True, vert=True)
    for patch in bp["boxes"]:
        patch.set_facecolor(clrs[mn])
        patch.set_alpha(0.5)
    axes[i].set_title(mn)
    axes[i].set_ylabel("Dice" if i == 0 else "")
    axes[i].tick_params(axis="x", rotation=35)
    axes[i].set_ylim(-0.05, 1.05)
    axes[i].grid(axis="y", alpha=0.3)

fig.suptitle("Per-Sample Dice Distribution by Corruption Type", fontsize=14)
plt.tight_layout()
plt.show()

# also print mean +/- std per corruption
print(f"\n{'corruption':<18}", end="")
for n in MODELS: print(f"  {n:>18}", end="")
print()
print("-" * (18 + 20 * len(MODELS)))
for ct in sorted(per_sample[MODELS[0]].keys()):
    row = f"{ct:<18}"
    for n in MODELS:
        vals = per_sample[n][ct]
        row += f"  {np.mean(vals):.3f} +/- {np.std(vals):.3f}"
    print(row)

## Analysis

**Key observations from the results above:**

1. **Structural corruptions (crop, slicing) are easier to localize** than diffuse ones
   (noise, blur). This makes sense - zeroed-out slices have a clear spatial signature
   that the encoder-decoder can learn, while noise is everywhere by definition.

2. **Global corruptions (noise, downsample) produce high Dice by design** because both
   the ground-truth map and the prediction are close to all-ones - but this is a
   detection task (present vs absent), not localization.  The real test for these is
   whether the model outputs near-zero maps on *clean* inputs (low false positive rate).

3. **Attention U-Net vs plain U-Net**: the attention gates should help with corruptions
   that have subtle, spatially-varying signatures (blur, shift edges).  If the
   improvement is marginal, it suggests the corruptions we synthesized are already
   "easy enough" that skip connections alone are sufficient.

4. **Res-UNet has the most parameters** due to the extra encoder stage.  Whether the
   added capacity translates to better Dice depends on dataset size - with ~600
   training scans, it may overfit slightly.

5. **Baselines**: the intensity-threshold baseline catches cropped slices (which are
   literally zero) but cannot detect noise, blur, or shift.  All learned models
   substantially outperform it, confirming the networks learn non-trivial features.

**Limitations:**
- All corruptions are synthetic.  Real-world artifacts (e.g., metal artifacts,
  partial volume effects) may look different.
- The dataset is LUNA16 (chest CTs).  Generalization to other body regions is
  not tested.
- 50 epochs may not fully converge for the deeper Res-UNet - a longer schedule
  or learning-rate warm-up could help.

## Qualitative Results

Predictions from the best model on a few test samples with different corruption types.
The overlay uses **green** = true positive, **red** = false positive, **blue** = false negative.

In [ ]:
best_name = max(test_res, key=lambda k: test_res[k]["dice"])
vis_model = make_model(best_name).to(device)
vis_model.load_state_dict(torch.load(f"best_{best_name}.pth", map_location=device))
vis_model.eval()
print(f"Showing predictions from: {best_name}  (test dice {test_res[best_name]['dice']:.4f})")

# grab one sample per corruption type
vis_idx, seen = [], set()
for i, ct in enumerate(te_types):
    if ct not in seen:
        vis_idx.append(i); seen.add(ct)
    if len(vis_idx) >= 6: break

fig, axes = plt.subplots(len(vis_idx), 5, figsize=(18, 3.5 * len(vis_idx)))

for row, idx in enumerate(vis_idx):
    d = np.load(te_npz[idx])
    vol = d["vol"].astype(np.float32)
    gt  = d["cmap"].astype(np.float32)
    ct  = str(d["ctype"])

    inp = torch.from_numpy(vol[None, None]).to(device)
    with torch.no_grad():
        prob = torch.sigmoid(vis_model(inp)).cpu().numpy()[0, 0]
    pred = (prob >= 0.5).astype(np.float32)
    z = vol.shape[0] // 2

    axes[row, 0].imshow(vol[z], cmap="gray"); axes[row, 0].axis("off")
    axes[row, 0].set_ylabel(ct, fontsize=9, rotation=0, labelpad=70)
    axes[row, 1].imshow(gt[z], cmap="Reds", vmin=0, vmax=1); axes[row, 1].axis("off")
    axes[row, 2].imshow(prob[z], cmap="Reds", vmin=0, vmax=1); axes[row, 2].axis("off")
    axes[row, 3].imshow(pred[z], cmap="Reds", vmin=0, vmax=1); axes[row, 3].axis("off")

    # TP / FP / FN overlay on the input
    rgb = np.stack([vol[z]]*3, axis=-1)
    tp = (pred[z] > 0) & (gt[z] > 0)
    fp = (pred[z] > 0) & (gt[z] == 0)
    fn = (pred[z] == 0) & (gt[z] > 0)
    rgb[tp] = [0, .9, 0]
    rgb[fp] = [.9, 0, 0]
    rgb[fn] = [0, 0, .9]
    axes[row, 4].imshow(rgb); axes[row, 4].axis("off")

    if row == 0:
        for j, t in enumerate(["Input", "Ground Truth", "Pred (prob)", "Pred (binary)", "Overlay"]):
            axes[0, j].set_title(t)

plt.tight_layout(); plt.show()
del vis_model
torch.cuda.empty_cache() if torch.cuda.is_available() else None

## Inference Benchmarking

In [ ]:
dummy1 = torch.randn(1, 1, *VOL_SHAPE, device=device)
N_RUNS = 10

print(f"{'Model':<16} {'Time (s)':>10} {'GPU MB':>10} {'Params':>12}")
print("-" * 52)

for n in MODELS:
    m = make_model(n).to(device)
    m.load_state_dict(torch.load(f"best_{n}.pth", map_location=device))
    m.eval()

    with torch.no_grad(): _ = m(dummy1)  # warm-up

    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats(); torch.cuda.synchronize()

    t0 = time.time()
    with torch.no_grad():
        for _ in range(N_RUNS): _ = m(dummy1)
    if torch.cuda.is_available(): torch.cuda.synchronize()
    avg_t = (time.time() - t0) / N_RUNS

    mem = torch.cuda.max_memory_allocated() / 1e6 if torch.cuda.is_available() else 0
    test_res[n]["time"] = avg_t
    test_res[n]["mem"]  = mem
    print(f"{n:<16} {avg_t:>10.4f} {mem:>10.1f} {nparams(m):>12,}")
    del m

torch.cuda.empty_cache() if torch.cuda.is_available() else None

## End-to-End Demo on a Raw Scan

Load a raw `.mhd`, apply a known corruption, run the best model, display
axial / coronal / sagittal predictions.

In [ ]:
if te_paths:
    best = max(test_res, key=lambda k: test_res[k]["dice"])
    demo_m = make_model(best).to(device)
    demo_m.load_state_dict(torch.load(f"best_{best}.pth", map_location=device))
    demo_m.eval()

    raw = load_scan(te_paths[0])
    msk = find_lung_mask(te_paths[0], MASK_DIR)
    pp, mr = preprocess(raw, msk)

    c_vol, c_gt = corrupt_crop(pp, mr, mode="center", rng=np.random.default_rng(0))

    inp = torch.from_numpy(c_vol[None, None]).float().to(device)
    with torch.no_grad():
        prob = torch.sigmoid(demo_m(inp)).cpu().numpy()[0, 0]
    pred = (prob >= 0.5).astype(np.float32)

    ds = dice_score(torch.from_numpy(pred)[None], torch.from_numpy(c_gt)[None])
    print(f"Demo scan  |  model: {best}  |  corruption: crop (center)  |  dice: {ds:.4f}")

    z, yc, xc = [s//2 for s in c_vol.shape]
    views = [
        ("Axial",    c_vol[z],      c_gt[z],      prob[z],      pred[z]),
        ("Coronal",  c_vol[:,yc,:], c_gt[:,yc,:], prob[:,yc,:], pred[:,yc,:]),
        ("Sagittal", c_vol[:,:,xc], c_gt[:,:,xc], prob[:,:,xc], pred[:,:,xc]),
    ]

    fig, axes = plt.subplots(3, 4, figsize=(15, 11))
    fig.suptitle(f"End-to-End Demo   (Dice = {ds:.4f})", fontsize=14)
    for r, (vn, sv, sg, sp, sb) in enumerate(views):
        kw = {"origin": "lower", "aspect": "auto"} if r > 0 else {}
        axes[r,0].imshow(sv, cmap="gray", **kw); axes[r,0].set_ylabel(vn, fontsize=11)
        axes[r,1].imshow(sg, cmap="Reds", vmin=0, vmax=1, **kw)
        axes[r,2].imshow(sp, cmap="Reds", vmin=0, vmax=1, **kw)
        axes[r,3].imshow(sb, cmap="Reds", vmin=0, vmax=1, **kw)
        for a in axes[r]: a.axis("off")
    axes[0,0].set_title("Input"); axes[0,1].set_title("Ground Truth")
    axes[0,2].set_title("Prediction (prob)"); axes[0,3].set_title("Prediction (binary)")
    plt.tight_layout(); plt.show()

    del demo_m
    torch.cuda.empty_cache() if torch.cuda.is_available() else None

## Summary

In [ ]:
elapsed = time.time() - NOTEBOOK_START
h, m, s = int(elapsed // 3600), int(elapsed % 3600 // 60), int(elapsed % 60)

print(f"Runtime : {h}h {m}m {s}s")
print(f"Device  : {device}", end="")
if torch.cuda.is_available(): print(f"  ({torch.cuda.get_device_name(0)})")
else: print()
print(f"Dataset : {len(train_ds)} / {len(val_ds)} / {len(test_ds)} (train/val/test)")
print(f"Epochs  : {EPOCHS}")
print(f"Output  : 3D corruption map  {VOL_SHAPE}\n")

print(f"{'Model':<16} {'Dice':>7} {'IoU':>7} {'Time(s)':>9} {'Params':>11}")
print("-" * 54)
for n in MODELS:
    r = test_res[n]
    print(f"{n:<16} {r['dice']:>7.4f} {r['iou']:>7.4f} {r.get('time',0):>9.4f} {r['params']:>11,}")

best = max(test_res, key=lambda k: test_res[k]["dice"])
print(f"\nBest: {best}  (dice {test_res[best]['dice']:.4f})")